In [50]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import pandas as pd

In [51]:
df_swaption = pd.read_excel('swaption_vol_data_2025-06-30.xlsx', sheet_name='bloomberg vcub')
df_swaption
df_swaption.head()

,reference,instrument,model,date,expiration,tenor,-200,-100,-50,-25,0,25,50,100,200
0,SOFR,swaption,black,2025-06-30,1,1,72.250,46.870,39.100,36.000,33.39,31.300,29.760,28.17,28.660
1,SOFR,swaption,black,2025-06-30,1,2,65.780,44.400,37.970,35.460,33.39,31.750,30.530,29.19,29.300
2,SOFR,swaption,black,2025-06-30,1,3,57.870,40.610,35.560,33.650,32.11,30.920,30.060,29.14,29.290
3,SOFR,swaption,black,2025-06-30,1,4,54.405,38.565,33.925,32.195,30.83,29.805,29.095,28.43,28.885
4,SOFR,swaption,black,2025-06-30,1,5,50.940,36.520,32.290,30.740,29.55,28.690,28.130,27.72,28.480


In [52]:
df_cap = pd.read_excel('cap_curves_2025-06-30.xlsx', sheet_name='rate curves 2025-06-30')
df_cap.head()

,tenor,swap rates,spot rates,discounts,forwards,flat vols,fwd vols
0,0.25,0.042353,0.042353,0.989523,NaN,NaN,NaN
1,0.50,0.040859,0.040852,0.979883,0.039351,0.156842,0.156842
2,0.75,0.039391,0.039372,0.971043,0.036414,0.180709,0.201708
3,1.00,0.038115,0.038083,0.962807,0.034217,0.204576,0.240464
4,1.25,0.036704,0.036653,0.955417,0.030938,0.242127,0.328341


# Q 1.1)

In [53]:
delta = 0.25
T_start = 1.0
T_end = 5.0   # 1 + 4 years

# ----------------------------
# Extract discount curve
# ----------------------------
def get_discount(t):
    return df_cap.loc[df_cap["tenor"] == t, "discounts"].iloc[0]

# Discount factors at swap boundaries
P_start = get_discount(T_start)
P_end = get_discount(T_end)

# Payment schedule (quarterly)
payment_tenors = np.arange(T_start + delta, T_end + delta/2, delta)

# Annuity denominator
annuity = 0

for t in payment_tenors:
    annuity += delta * get_discount(t)

# Forward swap rate
forward_swap_rate = (P_start - P_end) / annuity

print("1Y forward 4Y swap rate:", forward_swap_rate*100, "%")

1Y forward 4Y swap rate: 3.269770231881184 %


# Q 1.2)

In [54]:
df_swaption

,reference,instrument,model,date,expiration,tenor,-200,-100,-50,-25,0,25,50,100,200
0,SOFR,swaption,black,2025-06-30,1,1,72.250,46.870,39.100,36.000,33.39,31.300,29.760,28.17,28.660
1,SOFR,swaption,black,2025-06-30,1,2,65.780,44.400,37.970,35.460,33.39,31.750,30.530,29.19,29.300
2,SOFR,swaption,black,2025-06-30,1,3,57.870,40.610,35.560,33.650,32.11,30.920,30.060,29.14,29.290
3,SOFR,swaption,black,2025-06-30,1,4,54.405,38.565,33.925,32.195,30.83,29.805,29.095,28.43,28.885
4,SOFR,swaption,black,2025-06-30,1,5,50.940,36.520,32.290,30.740,29.55,28.690,28.130,27.72,28.480


In [55]:
import numpy as np
from scipy.stats import norm

# ----------------------------
# Parameters
# ----------------------------
delta = 0.25
notional = 100

expiry = 1
tenor = 4

# Forward swap rate (ATM forward)
F = forward_swap_rate

# ----------------------------
# Extract swaption row (expiration=1, tenor=4)
# ----------------------------
row = df_swaption[
    (df_swaption["expiration"] == expiry) &
    (df_swaption["tenor"] == tenor)
].iloc[0]

# Volatility strike columns
strike_cols = [-200,-100,-50,-25,0,25,50,100,200]

# ----------------------------
# Annuity factor
# ----------------------------
def discount(t):
    return df_cap.loc[df_cap["tenor"] == t, "discounts"].iloc[0]

def swap_annuity(start, end):
    schedule = np.arange(start + delta, end + 1e-12, delta)
    return sum(delta * discount(t) for t in schedule)

swap_end = expiry + tenor
A = swap_annuity(expiry, swap_end)

# ----------------------------
# Pricing table
# ----------------------------
results = []

for bp in strike_cols:

    # ATM-relative strike
    K = F + bp * 1e-4

    # Volatility (convert to decimal)
    sigma = row[(bp)] / 100

    if sigma <= 0 or K <= 0:
        price = np.nan
    else:
        T = expiry
        sqrtT = np.sqrt(T)

        d1 = (np.log(F/K) + 0.5*sigma**2*T) / (sigma*sqrtT)
        d2 = d1 - sigma*sqrtT

        price = notional * A * (F*norm.cdf(d1) - K*norm.cdf(d2))

    results.append({
        "strike_bp": bp,
        "strike_rate": K,
        "vol": sigma,
        "price": price
    })

swaption_price_1y4y = pd.DataFrame(results)

print(swaption_price_1y4y)

   strike_bp  strike_rate      vol     price
0       -200     0.012698  0.54405  7.271096
1       -100     0.022698  0.38565  3.948049
2        -50     0.027698  0.33925  2.536235
3        -25     0.030198  0.32195  1.943130
4          0     0.032698  0.30830  1.443366
5         25     0.035198  0.29805  1.042391
6         50     0.037698  0.29095  0.736560
7        100     0.042698  0.28430  0.355738
8        200     0.052698  0.28885  0.087983


# Q 1.3)

In [56]:
# ----------------------------
# Parameters
# ----------------------------
notional = 100
delta = 0.25

# ATM forward swap rate
F = forward_swap_rate

# Use ATM implied vol from 1Y x 4Y swaption
atm_row = df_swaption[
    (df_swaption["expiration"] == 1) &
    (df_swaption["tenor"] == 4)
].iloc[0]

atm_vol = atm_row[0] / 100   # vol is in %

# Swaptions to compare
swaption_list = [
    (0.25, 4, "3mo x 4yr"),
    (1, 4, "1yr x 4yr"),
    (2, 4, "2yr x 4yr"),
    (1, 2, "1yr x 2yr")
]

# ----------------------------
# Discount factor helper
# ----------------------------
def discount(t):
    return df_cap.loc[df_cap["tenor"] == t, "discounts"].iloc[0]

def swap_annuity(start, end):
    schedule = np.arange(start + delta, end + 1e-12, delta)
    return sum(delta * discount(t) for t in schedule)

# ----------------------------
# Pricing
# ----------------------------
results = []

for expiry, tenor, label in swaption_list:

    swap_end = expiry + tenor

    # ATM strike
    K = F

    # Use ATM vol from 1Y×4Y surface (as instructed)
    sigma = atm_vol

    T = expiry
    A = swap_annuity(expiry, swap_end)

    sqrtT = np.sqrt(T)

    if sigma <= 0:
        price = np.nan
    else:
        d1 = (np.log(F/K) + 0.5*sigma**2*T) / (sigma*sqrtT)
        d2 = d1 - sigma*sqrtT

        price = notional * A * (F*norm.cdf(d1) - K*norm.cdf(d2))

    results.append({
        "Swaption": label,
        "Expiration": expiry,
        "Tenor": tenor,
        "Price": price
    })

comparison_df = pd.DataFrame(results)

print(comparison_df)

    Swaption  Expiration  Tenor     Price
0  3mo x 4yr        0.25      4  0.741695
1  1yr x 4yr        1.00      4  1.443366
2  2yr x 4yr        2.00      4  1.966195
3  1yr x 2yr        1.00      2  0.745271


## Comparison of Swaption Prices Relative to 1Y × 4Y ATM Swaption

### Reference price
- 1Y × 4Y ATM swaption price = **1.443366**

### Other swaption prices

| Swaption | Price | Difference from 1Y×4Y | Price Ratio |
|---|---|---|---|
| 3mo × 4yr | 0.741695 | −0.701671 | 0.514 |
| 2yr × 4yr | 1.966195 | +0.522829 | 1.362 |
| 1yr × 2yr | 0.745271 | −0.698095 | 0.516 |

---

### Interpretation

1. **Expiration effect**
   - Increasing option expiration generally increases swaption value because the holder has more time for interest rate uncertainty to materialize.

2. **Tenor effect**
   - Reducing swap tenor reduces the underlying swap annuity, lowering option payoff magnitude.

3. **Most expensive swaption**
   - The 2yr × 4yr swaption is more expensive than the 1yr × 4yr swaption.
   - This occurs because longer option maturity increases time value even though volatility is held fixed.

---